# Create Target And Modeling Dataset V1

Goal: create a clean dataset where each row is one order and each delivered order has a correct SLA breach label.

Output: `data/processed/modeling_dataset_v1.csv`

## Logic

- Load `data/processed/base_order_dataset.csv`.
- Convert order date columns with `pd.to_datetime()`.
- Keep only delivered orders with actual and estimated delivery dates.
- Create `sla_breached = 1` when actual delivery is later than estimated delivery, else `0`.
- Create time-based columns for modeling and EDA.
- Save one row per order to `data/processed/modeling_dataset_v1.csv`.

`actual_delivery_days` and `delivery_delay_days` are retained for EDA only. They should not be used as model inputs because they depend on the actual delivery date.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features.create_target import (
    load_base_order_dataset,
    create_modeling_dataset,
    save_modeling_dataset,
)

In [2]:
input_path = PROJECT_ROOT / "data" / "processed" / "base_order_dataset.csv"
output_path = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_v1.csv"

base_orders = load_base_order_dataset(input_path)
base_orders.shape

(99441, 31)

In [3]:
modeling_dataset = create_modeling_dataset(base_orders)

print(f"Rows: {modeling_dataset.shape[0]:,}")
print(f"Columns: {modeling_dataset.shape[1]:,}")
print(f"Duplicate order_id rows: {modeling_dataset['order_id'].duplicated().sum():,}")
print(modeling_dataset['sla_breached'].value_counts().sort_index())

modeling_dataset.head()

Rows: 96,470
Columns: 38
Duplicate order_id rows: 0
sla_breached
0    88644
1     7826
Name: count, dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,avg_product_height_cm,avg_product_width_cm,avg_product_volume_cm3,sla_breached,estimated_delivery_days,actual_delivery_days,delivery_delay_days,purchase_day_of_week,purchase_hour,is_weekend_order
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,...,9.0,14.0,3528.0,0,15.625671,7.614421,-8.011250,2,8,0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,...,30.0,40.0,60000.0,0,18.546458,16.216181,-2.330278,2,10,0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,...,13.0,33.0,14157.0,0,21.393391,7.948437,-13.444954,6,14,1
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,12952,...,10.0,15.0,2400.0,0,11.582928,6.147269,-5.435660,2,10,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,13226,...,40.0,30.0,42000.0,0,40.418160,25.114352,-15.303808,5,13,1


In [4]:
assert modeling_dataset["order_id"].is_unique
assert set(modeling_dataset["order_status"].unique()) == {"delivered"}
assert modeling_dataset["order_delivered_customer_date"].notna().all()
assert modeling_dataset["order_estimated_delivery_date"].notna().all()
assert set(modeling_dataset["sla_breached"].unique()).issubset({0, 1})

save_modeling_dataset(modeling_dataset, output_path)
output_path

WindowsPath('d:/ML Project/deliveriq-sla-breach-prediction/data/processed/modeling_dataset_v1.csv')